# 🌸 Iris Dataset Overview

The **Iris dataset** is one of the most well-known datasets in machine learning, used often for classification tasks and algorithm testing.

## 📋 Description

- The dataset contains **150 samples** from three species of Iris flowers:
  - *Iris-setosa*
  - *Iris-versicolor*
  - *Iris-virginica*
- Each sample has **4 features** (numeric, continuous):
  1. **Sepal length** (in cm)
  2. **Sepal width** (in cm)
  3. **Petal length** (in cm)
  4. **Petal width** (in cm)

## 🧾 Structure

| Feature         | Description              | Data Type |
|-----------------|--------------------------|-----------|
| Sepal Length    | Length of the sepal      | float     |
| Sepal Width     | Width of the sepal       | float     |
| Petal Length    | Length of the petal      | float     |
| Petal Width     | Width of the petal       | float     |
| Target (Label)  | Iris species (0,1,2)     | integer   |

## 🎯 Objective

Given the four input features, the goal is to classify each sample into one of the **three Iris species**. This is a **multi-class classification** problem.

## 📊 Class Distribution

- Each class has **50 samples**, so the dataset is **balanced**.
  - Setosa: 50
  - Versicolor: 50
  - Virginica: 50

## 📚 Applications

- Algorithm testing and benchmarking
- Introduction to supervised learning
- Visualization of decision boundaries
- Feature importance analysis

## 📦 Source

Originally introduced by **Ronald A. Fisher** in 1936 in the paper *"The use of multiple measurements in taxonomic problems"*. It is now available in `sklearn.datasets` as a built-in toy dataset.



In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load Iris dataset
iris = load_iris()          
X = iris.data  # Features (150, 4)
y = iris.target  # Labels (0,1,2)

In [3]:
X.shape, y.shape

((150, 4), (150,))

In [4]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [6]:
# Define a simple feedforward network
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(4, 16)   # input layer to hidden layer
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 3)   # hidden layer to output layer (3 classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x  # raw logits (no softmax here, as CrossEntropyLoss expects logits)

In [7]:
# Instantiate the model, define loss and optimizer
model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [8]:
# Training loop
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [20/100], Loss: 0.4264
Epoch [40/100], Loss: 0.2264
Epoch [60/100], Loss: 0.1213
Epoch [80/100], Loss: 0.0765
Epoch [100/100], Loss: 0.0595


In [9]:
# Evaluation
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    acc = accuracy_score(y_test, predicted)
    print(f'Accuracy on test set: {acc*100:.2f}%')

Accuracy on test set: 96.67%


### 🔍 Explanation of the Line:
```python
return x  # raw logits (no softmax here, as CrossEntropyLoss expects logits)
```

---

### 📌 Why return raw logits?

In PyTorch, when using `nn.CrossEntropyLoss`, the model should return **raw logits**, not probabilities.

- **Logits** = raw scores output by the final layer (no softmax applied)
- `CrossEntropyLoss` automatically applies `softmax` followed by `log()` and `NLLLoss` internally.

---

### ❓ Why not apply `softmax` manually?

If you apply `softmax` in your model **before** passing it to `CrossEntropyLoss`, you're applying `softmax` **twice**, which:

- Skews gradient calculations
- Leads to incorrect loss values
- Slows or breaks training

---

### ✅ Best Practice

Just return the raw output (`x`) from the model's final layer and let `nn.CrossEntropyLoss` handle softmax internally.

---

### 🧠 Summary

| Term              | Explanation                                                  |
|-------------------|--------------------------------------------------------------|
| **Logits**        | Raw output from the model's last layer (no softmax)          |
| **Softmax**       | Converts logits to probabilities (internally done)           |
| **CrossEntropyLoss** | Combines softmax + log + NLLLoss automatically            |

---
**Therefore:**

```python
return x  # raw logits (no softmax here, as CrossEntropyLoss expects logits)
```

✔️ This is the correct approach for multi-class classification with `CrossEntropyLoss`.
